# Keeping the peak day: sizing a grid connection from typical days

A region's grid connection has to carry its highest load. Suppose you size it with an
optimization model that runs on typical days instead of every hour. Then the typical days must
contain that peak: if their highest hour is lower, the grid connection comes out too small.

In this tutorial we follow the peak through four steps on six weeks of hourly data. By the end
you will have seen that

* the typical days miss the peak, because the peak day shares its typical day with days of lower
  load,
* more typical days don't bring it back,
* `ExtremeConfig` keeps it exactly, for one extra typical day, and
* segments flatten it again.

## 1  The default misses the peak

We use the six weeks of hourly data from [Your first aggregation](../quickstart.ipynb): solar
irradiance, temperature, wind and load. Every run reduces them to typical days of 24 hours, so we
write the settings down once in a small function. A second function reads the highest load of the
typical days, the peak the model sees.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam
from tsam import ExtremeConfig, SegmentConfig

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data


def aggregate(n_clusters=6, **options):
    return tsam.aggregate(
        data,
        n_clusters=n_clusters,
        period_duration="1D",
        numerical_tolerance=1e-9,  # tolerate rounding noise in tsam's range check
        **options,
    )


def highest_hour(result):
    return result.cluster_representatives["Load"].max()

The highest load of the six weeks, which the grid connection must carry:

In [ ]:
peak_hour = data["Load"].idxmax()
print(f"{data['Load'].max():.1f} MW on {peak_hour:%d %B}")

Now 6 typical days with tsam's defaults:

In [ ]:
result = aggregate()
print(f"highest typical hour: {highest_hour(result):.1f} MW")

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert (
    peak_hour.strftime("%d %B") == "04 February"
    and round(data["Load"].max(), 1) == 636.5
)
assert round(highest_hour(result), 1) == 601.5

601.5 MW instead of 636.5 MW: a grid connection sized on these typical days would be 35 MW too
small. Look at the group of days that holds the peak. Its days are the faint lines, and its
typical day is the bold line:

In [ ]:
peak_day = (peak_hour - data.index[0]).days  # the day's position in the six weeks
peak_group = int(result.cluster_assignments[peak_day])
result.plot.cluster_members(
    columns=["Load"],
    clusters=[peak_group],
    units={"Load": "MW"},
    title="The group of 4 February",
)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
members = result.cluster_assignments == peak_group
assert members.sum() == 10
assert data.loc[f"{peak_hour:%Y-%m-%d}", "Load"].max() > highest_hour(result) + 30

The bold typical day stands in for all ten days of the group, and the peak day is one of them.
Its highest hours lie above the typical day, so the model never sees them.

## 2  More typical days?

With more typical days, each group gets smaller. Does the peak come back?

In [ ]:
pd.Series(
    {n: highest_hour(aggregate(n)) for n in [6, 8, 12, 16, 20]},
    name="highest typical hour (MW)",
).rename_axis("typical days").round(1)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
more = pd.Series({n: highest_hour(aggregate(n)) for n in [6, 8, 12, 16, 20]})
assert more.is_monotonic_increasing
assert more.max() < data["Load"].max() - 15
assert round(data["Load"].max() - more[20]) == 18

The peak creeps up, but even 20 typical days, more than three times as many, stay 18 MW short.
However small the groups get, the peak day keeps sharing its typical day with days of lower load.

## 3  Name the day

When the model needs one particular day as it is, name it. `ExtremeConfig(max_value=["Load"])`
takes the day with the highest load out of its group and adds it as a typical day of its own:

In [ ]:
kept = aggregate(extremes=ExtremeConfig(max_value=["Load"]))
print(
    f"typical days: {kept.n_clusters}, highest typical hour: {highest_hour(kept):.1f} MW"
)
print(
    f"RMSE: {result.accuracy.weighted_rmse:.3f} before, {kept.accuracy.weighted_rmse:.3f} now"
)
kept.plot.cluster_representatives(
    columns=["Load"], units={"Load": "MW"}, title="Six typical days and the peak day"
)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert kept.n_clusters == 7 and sorted(kept.cluster_counts.values())[0] == 1
assert round(highest_hour(kept), 1) == round(data["Load"].max(), 1)
assert (
    round(kept.accuracy.weighted_rmse, 3)
    == round(result.accuracy.weighted_rmse, 3)
    == 0.123
)

Seven typical days now, and the highest typical hour is exactly the peak, 636.5 MW. The plot
labels each typical day with the number of days it stands for: the new one stands for a single
day, 4 February. The other six fit the remaining days as before, and the RMSE, how far the
typical days are from the real days hour by hour, stays at 0.123.

## 4  Segments flatten it again

Segments merge neighbouring hours of each day into fewer timesteps, which makes the model smaller.
Add 4 segments to the configuration of step 3 and compare the peak day with and without them:

In [ ]:
segmented = aggregate(
    extremes=ExtremeConfig(max_value=["Load"]), segments=SegmentConfig(n_segments=4)
)
peak_date = f"{peak_hour:%Y-%m-%d}"
peak_day_load = pd.DataFrame(
    {
        "every hour": kept.reconstructed.loc[peak_date, "Load"],
        "4 segments": segmented.reconstructed.loc[peak_date, "Load"],
    }
)
fig = px.line(
    peak_day_load, line_shape="hv", title="4 February: every hour vs. 4 segments"
)
fig.update_yaxes(title_text="Load [MW]")
fig.update_layout(legend_title_text="")
fig.show()

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert len(peak_day_load) == 24
assert round(peak_day_load["every hour"].max(), 1) == round(data["Load"].max(), 1)
assert peak_day_load["4 segments"].nunique() == 4
assert round(highest_hour(segmented)) == round(peak_day_load["4 segments"].max()) == 617

Each segment is one flat step. The highest step holds the average of the hours around the peak,
617 MW instead of 636.5 MW. The fewer the segments, the more of the peak is lost:

In [ ]:
pd.Series(
    {
        n: highest_hour(
            aggregate(
                extremes=ExtremeConfig(max_value=["Load"]),
                segments=SegmentConfig(n_segments=n),
            )
        )
        for n in [24, 12, 6, 4]
    },
    name="highest typical hour (MW)",
).rename_axis("segments per day").round(1)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
fewer = pd.Series(
    {
        n: highest_hour(
            aggregate(
                extremes=ExtremeConfig(max_value=["Load"]),
                segments=SegmentConfig(n_segments=n),
            )
        )
        for n in [24, 12, 6, 4]
    }
)
assert round(fewer[24], 1) == round(data["Load"].max(), 1)
assert fewer.is_monotonic_decreasing
assert (
    round(data["Load"].max() - fewer[12]) == 5
    and round(data["Load"].max() - fewer[4]) == 20
)

With 12 segments the model has half the timesteps and the peak is 5 MW short; with 4 segments,
20 MW. Whatever makes the model smaller can cost the peak again, so check it after every such
change.

## What you've learned

* A typical day stands in for its whole group. When the peak day shares its group with days of
  lower load, its highest hours disappear.
* More typical days make the groups smaller, but don't bring back one particular day.
* `ExtremeConfig` keeps the day exactly, for one more typical day, and the other typical days fit
  the remaining days as before.
* Segments average neighbouring hours and flatten the peak again. Check the peak after every
  change that makes the model smaller.

Where to go next:

* [Extreme periods](../../how-to/extreme_periods.ipynb) — the other ways to pick a day, and the
  other ways to add it to the result.
* [Comparing clustering methods](../comparing_clustering_methods.ipynb) — `kmaxoids`, which gives
  the most unusual day a group of its own.
* [Comparing representations](../comparing_representations.ipynb) — `maxoid` and `minmax_mean`,
  which move typical days towards their extremes.
* [Segmentation](../../how-to/segmentation.ipynb) — choose the number of segments.
* [Extreme periods](../../explanation/how-aggregation-works/04_extreme_periods.ipynb), explained —
  how the day is found and added, step by step.